In [ ]:
!pip install pyautogen google-generativeai --quiet

import google.generativeai as genai
from autogen import AssistantAgent, UserProxyAgent, GroupChat, GroupChatManager
import os
import json
from typing import Dict

# Configure API and model
class HealthConfig:
    def __init__(self, model_name: str = "gemini-pro"):
        self.api_key = os.getenv("GEMINI_API_KEY", "your_gemini_api_key")
        if not self.api_key or self.api_key == "your_gemini_api_key":
            raise EnvironmentError("GEMINI_API_KEY not set")
        genai.configure(api_key=self.api_key)
        self.llm_config = {
            "config_list": [
                {
                    "model": model_name,
                    "api_key": self.api_key,
                    "api_type": "google"
                }
            ],
            "temperature": 0.6,  # Changed from 0.7
            "max_tokens": 1000   # Added token limit
        }

# BMI calculation
def compute_bmi(weight_kg: float, height_cm: float) -> Dict:
    """Calculate BMI and categorize."""
    height_m = height_cm / 100
    bmi = weight_kg / (height_m ** 2)
    if bmi < 18.5:
        category = "Underweight"
    elif 18.5 <= bmi < 25:
        category = "Healthy"
    elif 25 <= bmi < 30:
        category = "Overweight"
    else:
        category = "Obese"
    return {
        "bmi": round(bmi, 2),
        "category": category,
        "weight_kg": weight_kg,
        "height_cm": height_cm
    }

# Agents
client = UserProxyAgent(
    name="Client",
    human_input_mode="NEVER",
    code_execution_config={"use_docker": False},
    system_message="You represent the user, coordinating the health planning process."
)

health_analyzer = AssistantAgent(
    name="Health_Analyzer",
    llm_config=HealthConfig().llm_config,
    system_message="""You are the Health Analysis Agent.
    Analyze BMI data and provide tailored health advice.
    BMI Categories:
    - Underweight: <18.5
    - Healthy: 18.5–24.9
    - Overweight: 25–29.9
    - Obese: ≥30
    Offer clear, supportive recommendations for lifestyle improvements."""
)

nutrition_coach = AssistantAgent(
    name="Nutrition_Coach",
    llm_config=HealthConfig().llm_config,
    system_message="""You are the Nutrition Coach Agent.
    Design a 1-day meal plan based on:
    - BMI category and health advice
    - Dietary preferences (Vegetarian/Non-Vegetarian/Vegan)
    - Age and gender
    Include breakfast, lunch, dinner, and one snack with portion sizes."""
)

fitness_coach = AssistantAgent(
    name="Fitness_Coach",
    llm_config=HealthConfig().llm_config,
    system_message="""You are the Fitness Coach Agent.
    Create a 1-week exercise plan based on:
    - BMI category and health goals
    - Age, gender, and fitness level
    Include:
    - Exercise types (cardio, strength, flexibility)
    - Duration and intensity
    - Rest days and beginner modifications"""
)

# Group chat
health_group = GroupChat(
    agents=[client, health_analyzer, nutrition_coach, fitness_coach],
    messages=[],
    max_round=5,  # Changed from default
    speaker_selection_method="round_robin"
)

health_coordinator = GroupChatManager(
    groupchat=health_group,
    llm_config=HealthConfig().llm_config,
    system_message="""You are the Health Coordinator.
    Manage agent interactions to create a comprehensive health plan.
    Workflow:
    1. Client provides user data
    2. Health Analyzer evaluates BMI
    3. Nutrition Coach designs meal plan
    4. Fitness Coach creates exercise plan"""
)

# Main function
def run_health_advisor():
    print("=== Health Advisor System ===")
    print("Create a personalized health plan.")

    try:
        # Collect user input
        weight_kg = float(input("Weight (kg): "))
        height_cm = float(input("Height (cm): "))
        age = int(input("Age: "))
        gender = input("Gender (M/F): ").strip().upper()
        diet_pref = input("Dietary Preference (Vegetarian/Non-Vegetarian/Vegan): ").strip()

        # Validate inputs
        if weight_kg <= 0 or height_cm <= 0 or age <= 0:
            raise ValueError("Weight, height, and age must be positive")
        if gender not in ['M', 'F']:
            gender = 'M'

        # Calculate BMI
        bmi_data = compute_bmi(weight_kg, height_cm)
        print(f"\n✅ BMI: {bmi_data['bmi']} ({bmi_data['category']})")

        # Group chat message
        initial_message = f"""
        Health Planning Request:
        - Weight: {bmi_data['weight_kg']} kg
        - Height: {bmi_data['height_cm']} cm
        - BMI: {bmi_data['bmi']} ({bmi_data['category']})
        - Age: {age}, Gender: {gender}
        - Diet Preference: {diet_pref}

        Health_Analyzer: Provide health advice.
        Nutrition_Coach: Create a 1-day meal plan.
        Fitness_Coach: Design a 1-week exercise plan.
        """

        print("\n🚀 Starting Health Planning...")
        result = client.initiate_chat(
            recipient=health_coordinator,
            message=initial_message,
            max_turns=3  # Changed from 1 per agent
        )

        # Save output to JSON
        output = {
            "bmi_data": bmi_data,
            "profile": {"age": age, "gender": gender, "diet_pref": diet_pref},
            "health_advice": health_analyzer.last_message().get("content", "No advice"),
            "meal_plan": nutrition_coach.last_message().get("content", "No meal plan"),
            "exercise_plan": fitness_coach.last_message().get("content", "No exercise plan")
        }
        with open("health_plan.json", "w") as f:
            json.dump(output, f, indent=2)

        print("\n" + "="*40)
        print("✅ Health Plan Generated!")
        print(f"📊 BMI: {bmi_data['bmi']} ({bmi_data['category']})")
        print(f"👤 Profile: {age} year old {gender}, {diet_pref}")
        print("📄 Plan saved to health_plan.json")

        return output

    except Exception as e:
        print(f"System error: {e}")
        return None

# Simplified fallback
def basic_health_advisor():
    print("=== Basic Health Advisor ===")
    try:
        weight_kg = 78
        height_cm = 165
        age = 22
        gender = "M"
        diet_pref = "Non-Vegetarian"

        bmi_data = compute_bmi(weight_kg, height_cm)
        print(f"\n✅ BMI: {bmi_data['bmi']} ({bmi_data['category']})")

        advice = {
            "Underweight": "Increase calorie intake with nutrient-rich foods and strength exercises.",
            "Healthy": "Maintain balanced diet and regular exercise.",
            "Overweight": "Focus on calorie control and increased cardio activity.",
            "Obese": "Seek professional guidance for weight management."
        }.get(bmi_data['category'], "Consult a healthcare provider.")

        output = {
            "bmi_data": bmi_data,
            "profile": {"age": age, "gender": gender, "diet_pref": diet_pref},
            "advice": advice,
            "meal_plan": "Sample: Breakfast (oatmeal), Lunch (grilled chicken salad), Dinner (baked fish), Snack (nuts)",
            "exercise_plan": "Sample: 3 days cardio (30 min), 2 days strength (20 min), 2 rest days"
        }

        with open("basic_health_plan.json", "w") as f:
            json.dump(output, f, indent=2)

        print(f"\n💡 Advice: {advice}")
        print(f"📄 Saved to basic_health_plan.json")
        return output

    except Exception as e:
        print(f"Fallback error: {e}")
        return None

if __name__ == "__main__":
    try:
        run_health_advisor()
    except Exception as e:
        print(f"Multi-agent error: {e}")
        print("Switching to basic advisor...")
        basic_health_advisor()